In [1]:
# Define the function to create and configure the job application
def create_application():
    """
    Create and configure the job application 
    """

    company_name = input("Enter the company name: ") # Prompt the user to enter the company name
    role = input("Enter the role you are applying for: ") # Prompt the user to enter the role they are applying for
    application_status = input("Enter the application status (e.g., Applied, Interviewing, Offer, Rejected): ") # Prompt the user to enter the application status
    timeframe = input("Enter the date applied (YYYY-MM-DD): ") # Prompt the user to enter the date applied

    application = {
        "company": company_name,
        "role": role,
        "status": application_status,
        "date": timeframe,
    }
    return application

# Test the function by calling it and printing the result
create_application()

# Initialize an empty list to store job applications
applications = []
applications.append(create_application()) # Call the function and add the returned application to the list
print(applications) # Print the list of job applications

In [2]:
# Define a function to collect multiple job applications from the user
def collect_applications():

    applications = []

    while True:
        applications.append(create_application()) # Call the function and add the returned application to the list

        another = input("Add another application? (y/n): ") # Prompt the user to add another application
        if another.lower() != "y":
            break

    return applications

# Test the function by calling it and printing the result
my_applications = collect_applications()
print(my_applications)

In [3]:
# Define a function to list all job applications
def list_applications(applications):
    if not applications:
        print("No applications to show yet.")
        return

    # APPENDIX: Print the list of job applications in a user-friendly format
    print("\nYour job applications:")
    for number, app in enumerate(applications, start=1):
        print(f"{number}. {app['company']} — {app['role']} — {app['status']} — {app['date']}")

# Test the function by calling it and printing the result
list_applications(my_applications)

In [4]:
# Install the python-docx library to work with Word documents
#!pip install python-docx

import docx
print(docx.__version__)

1.2.0


In [ ]:
# Define a function to save the job applications to a Word document
import docx
from datetime import datetime

def save_report(applications):

    doc = docx.Document()

    today = datetime.now().strftime("%Y-%m-%d")

    doc.add_heading("My job applications Report", level=0)
    doc.add_paragraph(f"Report generated on {today}")

    # APPENDIX: Create a table with a header row
    table = doc.add_table(rows=1, cols=4)
    table.style = "Table Grid"

    # APPENDIX: Add headers to the table and make them bold
    headers = ["Company", "Role", "Status", "Date Applied"]
    hdr_cells = table.rows[0].cells
    for i, heading in enumerate(headers):
        # Make each header cell bold
        run = hdr_cells[i].paragraphs[0].add_run(heading)
        run.bold = True

    
    for app in applications: # # Add one row per application
        row_cells = table.add_row().cells
        row_cells[0].text = app["company"]
        row_cells[1].text = app["role"]
        row_cells[2].text = app["status"]
        row_cells[3].text = app["date"]

    filename = f"job_applications_{today}.docx"
    doc.save(filename)
    print(f"Report saved as {filename}")

    return filename # Test the function by calling it and printing the result

In [14]:
save_report(load_applications())

Loaded 7 my applications from applications.json
Report saved as job_applications_2026-07-28.docx


'job_applications_2026-07-28.docx'

# Save the report to a Word document
save_report(my_applications)

## Google API Authentication

This section uses the Gmail API to log in and send the report by email.
Authentication is handled via OAuth 2.0. The ** credentials.json ** file
(downloaded from Google Cloud) is read by `get_creds()`, and a `token.json`
is created on first login so the user does not log in every time.

Note: `credentials.json` and `token.json` are excluded from version control.

In [6]:
# Google API authentication
# Based on the Gmail API code provided in class (S. Weiss)
import os.path

from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError


def get_creds():
    """Authenticate with Google and return valid credentials.

    Reads credentials.json, opens a browser for login on first run,
    and saves token.json so future runs log in automatically.
    """
    SCOPES = ["https://www.googleapis.com/auth/gmail.modify",
              "https://www.googleapis.com/auth/gmail.send"]

    creds = None

    if os.path.exists("token.json"):
        creds = Credentials.from_authorized_user_file("token.json", SCOPES)
        print("Have tokens!")

    # If there are no (valid) credentials available, let the user log in.
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
            print("Refreshed token!")
        else:
            flow = InstalledAppFlow.from_client_secrets_file(
                "credentials.json", SCOPES
            )
            creds = flow.run_local_server(port=0)
            print("Got new token")

    # Save the credentials for the next run
    with open("token.json", "w") as token:
        token.write(creds.to_json())

    return creds

# Get credentials for Gmail API
creds = get_creds()

In [7]:
# Gmail API code to send an email
import base64
from email.message import EmailMessage
# Based on the Gmail API code provided in class (S. Weiss)
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

# Define a function to send an email using the Gmail API
def gmail_send_message(messageContent, subject, recipient, attachment=None):

    creds = get_creds()

    try:
        service = build("gmail", "v1", credentials=creds)
        message = EmailMessage()

        message.set_content(messageContent)
        message["To"] = recipient
        message["Subject"] = subject

        # Optional way to add an attachment to the email
        if attachment:
            with open(attachment, "rb") as content_file:
                content = content_file.read()
                message.add_attachment(
                    content,
                    maintype="application",
                    subtype=(attachment.split(".")[-1]),
                    filename=attachment,
                )

        encoded_message = base64.urlsafe_b64encode(message.as_bytes()).decode()
        create_message = {"raw": encoded_message}
        send_message = (
            service.users()
            .messages()
            .send(userId="me", body=create_message)
            .execute()
        )
        print(f'Message Id: {send_message["id"]}')
    except HttpError as error:
        print(f"An error occurred: {error}")
        send_message = None

    return send_message



# make sure the file exists first
report_file = save_report(my_applications)   

# Send the email with the report attached
gmail_send_message(
    messageContent="Here is my job applications report.",
    subject="My Job Applications",
    recipient="arkjedrzejewsky@gmail.com",
    attachment=report_file,
)

In [8]:
# Define a function to move the report file to a "logs" directory
import os
import shutil

def move_to_logs(filename):
    logs_folder = "logs"

    if not os.path.exists(logs_folder): #APPENDIX: Create the "logs" folder if it doesn't exist
        os.mkdir(logs_folder)

    destination = os.path.join(logs_folder, filename)
    shutil.move(filename, destination)
    print(f"Moved {filename} to {destination}")

    return destination

# Move the report file to the "logs" directory

report_file = save_report(my_applications)
move_to_logs(report_file)

In [9]:
# Define the main function to run the job application tracker menu
def main():

    applications = load_applications()   # APPENDIX: Load any previously saved applications at startup

    while True:
        print("\n Job Application Tracker")
        print("1. Add my applications")
        print("2. View all applications")
        print("3. Show statistics")
        print("4. Save and email report")
        print("5. Exit")

        my_choice = input("Choose an option (1-5): ")  # Prompt the user to choose an option from the menu

        if my_choice == "1":
            new_apps = collect_applications()            # APPENDIX: Collect new applications from the user
            applications = applications + new_apps       # APPENDIX: Add them to the existing list
            save_applications(applications)              # APPENDIX: Save the updated list to JSON
        elif my_choice == "2":
            list_applications(applications)
        # APPENDIX: Choice 3: Show statistics about the applications
        elif my_choice == "3":
            show_stats(applications)
        # APPENDIX: Choice 4: Save and email report
        elif my_choice == "4":
            report_file = save_report(applications)
            gmail_send_message(
                messageContent="Here is my job applications report.",
                subject="My Job Applications",
                recipient="arkjedrzejewsky@gmail.com",
                attachment=report_file,
            )
            move_to_logs(report_file)
        elif my_choice == "5":
            print("Goodbye!")
            break
        else:
            print("Invalid choice, please pick 1-5.")

In [10]:
# Define a function to save the list of applications to a JSON file
import json

def save_applications(applications, filename="applications.json"):
  
    with open(filename, "w") as f: #APPENDIX: Open the file in write mode and save the applications list as JSON
        json.dump(applications, f, indent=4)
    print(f"Saved {len(applications)} applications to {filename}")

# Another test to save the applications to a JSON file
save_applications(my_applications)

In [11]:
# Load applications from a JSON file if it exists
import os
import json

def load_applications(filename="applications.json"):

    if not os.path.exists(filename):
        print("You dont have any saved applications yet. Starting with an empty list.")
        return []

    with open(filename, "r") as f: #APPENDIX: Open the file in read mode and load the applications list from JSON
        applications = json.load(f)
    print(f"Loaded {len(applications)} my applications from {filename}")
    return applications

# Test the function by calling it and printing the result
loaded = load_applications()
list_applications(loaded)

In [12]:
# Show statistics about the applications
def show_stats(applications):

    if not applications:
        print("No applications to summarise yet.")
        return

    print(f"\nTotal applications: {len(applications)}")

    # APPENDIX: Count how many applications have each status
    status_counts = {}
    for app in applications:
        status = app["status"]
        if status in status_counts:
            status_counts[status] += 1
        else:
            status_counts[status] = 1

    print("Applications sorted by status:")
    for status, count in status_counts.items():
        print(f"  {status}: {count}")

# Test the function by calling it and printing the result
show_stats(my_applications)

In [13]:
# Run the main function to start the job application tracker
main()

Loaded 3 my applications from applications.json

 Job Application Tracker
1. Add my applications
2. View all applications
3. Show statistics
4. Save and email report
5. Exit
Saved 7 applications to applications.json

 Job Application Tracker
1. Add my applications
2. View all applications
3. Show statistics
4. Save and email report
5. Exit

Total applications: 7
Applications sorted by status:
  Applied: 3
  Offer: 3
  Interviewing: 1

 Job Application Tracker
1. Add my applications
2. View all applications
3. Show statistics
4. Save and email report
5. Exit

Total applications: 7
Applications sorted by status:
  Applied: 3
  Offer: 3
  Interviewing: 1

 Job Application Tracker
1. Add my applications
2. View all applications
3. Show statistics
4. Save and email report
5. Exit
Report saved as job_applications_2026-07-28.docx
Have tokens!
Refreshed token!
Message Id: 19fa821c63c675ca
Moved job_applications_2026-07-28.docx to logs/job_applications_2026-07-28.docx

 Job Application Tracker
